# Section A – RDD Creation and Partitioning

In [1]:
import pyspark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
0,application_1777732066215_0001,pyspark,idle,Link,Link,✔


SparkSession available as 'spark'.


In [9]:
numbers = list(range(1, 11))

rdd_numbers = sc.parallelize(numbers, 3)

print(rdd_numbers.collect())
print("Number of partitions:", rdd_numbers.getNumPartitions())

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
('Number of partitions:', 3)

In [10]:
rdd_numbers.glom().collect()

[[1, 2, 3], [4, 5, 6], [7, 8, 9, 10]]

# Section B – Text Analysis

In [11]:
book_rdd = sc.textFile("/user/hadoop/books.txt")

print("Number of partitions:", book_rdd.getNumPartitions())
print(book_rdd.take(5))

('Number of partitions:', 2)
[u'*** START OF THE PROJECT GUTENBERG EBOOK 1342 ***', u'', u'', u'', u'']

In [16]:
book_rdd_4 = sc.textFile("/user/hadoop/books.txt", 4)

print("Number of partitions:", book_rdd_4.getNumPartitions())

('Number of partitions:', 4)

In [17]:
book_rdd_4.glom().collect()

[[u'*** START OF THE PROJECT GUTENBERG EBOOK 1342 ***', u'', u'', u'', u'', u'                            [Illustration:', u'', u'                             GEORGE ALLEN', u'                               PUBLISHER', u'', u'                        156 CHARING CROSS ROAD', u'                                LONDON', u'', u'                             RUSKIN HOUSE', u'                                   ]', u'', u'                            [Illustration:', u'', u'               _Reading Jane\u2019s Letters._      _Chap 34._', u'                                   ]', u'', u'', u'', u'', u'                                PRIDE.', u'                                  and', u'                               PREJUDICE', u'', u'                                  by', u'                             Jane Austen,', u'', u'                           with a Preface by', u'                           George Saintsbury', u'                                  and', u'                           Illustrati

In [21]:
book_rdd = sc.textFile("/user/hadoop/books.txt")

In [22]:
words_rdd = book_rdd.flatMap(lambda line: line.split())

print(words_rdd.take(20))

[u'***', u'START', u'OF', u'THE', u'PROJECT', u'GUTENBERG', u'EBOOK', u'1342', u'***', u'[Illustration:', u'GEORGE', u'ALLEN', u'PUBLISHER', u'156', u'CHARING', u'CROSS', u'ROAD', u'LONDON', u'RUSKIN', u'HOUSE']

In [23]:
filtered_words = words_rdd.filter(lambda word: len(word) > 2)

print(filtered_words.take(20))

[u'***', u'START', u'THE', u'PROJECT', u'GUTENBERG', u'EBOOK', u'1342', u'***', u'[Illustration:', u'GEORGE', u'ALLEN', u'PUBLISHER', u'156', u'CHARING', u'CROSS', u'ROAD', u'LONDON', u'RUSKIN', u'HOUSE', u'[Illustration:']

In [25]:
clean_words = book_rdd.flatMap(lambda line: line.split()) \
    .map(lambda word: word.lower()) \
    .map(lambda word: ''.join([ch for ch in word if ch.isalpha()])) \
    .filter(lambda word: len(word) > 2)

print(clean_words.take(30))

[u'start', u'the', u'project', u'gutenberg', u'ebook', u'illustration', u'george', u'allen', u'publisher', u'charing', u'cross', u'road', u'london', u'ruskin', u'house', u'illustration', u'reading', u'janes', u'letters', u'chap', u'pride', u'and', u'prejudice', u'jane', u'austen', u'with', u'preface', u'george', u'saintsbury', u'and']

In [26]:
word_pairs = clean_words.map(lambda word: (word, 1))

word_counts = word_pairs.reduceByKey(lambda a, b: a + b)

print(word_counts.take(20))

[(u'cruscans', 2), (u'expostulation', 1), (u'desirable', 11), (u'foul', 1), (u'four', 36), (u'brightening', 1), (u'protest', 2), (u'sleep', 3), (u'greatgrandmother', 1), (u'consists', 2), (u'appetite', 2), (u'prognostics', 3), (u'xxi', 2), (u'looking', 30), (u'votes', 1), (u'pardon', 14), (u'sweetest', 3), (u'seriously', 13), (u'presents', 3), (u'neighbours', 11)]

In [27]:
print("Sample clean words:")
print(clean_words.take(20))

print("Sample word count pairs:")
print(word_counts.take(20))

Sample clean words:
[u'start', u'the', u'project', u'gutenberg', u'ebook', u'illustration', u'george', u'allen', u'publisher', u'charing', u'cross', u'road', u'london', u'ruskin', u'house', u'illustration', u'reading', u'janes', u'letters', u'chap']
Sample word count pairs:
[(u'cruscans', 2), (u'expostulation', 1), (u'desirable', 11), (u'foul', 1), (u'four', 36), (u'brightening', 1), (u'protest', 2), (u'sleep', 3), (u'greatgrandmother', 1), (u'consists', 2), (u'appetite', 2), (u'prognostics', 3), (u'xxi', 2), (u'looking', 30), (u'votes', 1), (u'pardon', 14), (u'sweetest', 3), (u'seriously', 13), (u'presents', 3), (u'neighbours', 11)]

# 9. Explore key-value dataset and retrieve meaningful subsets

In [28]:
love_words = word_counts.filter(lambda x: x[0].startswith("love"))

print(love_words.collect())

[(u'loveliest', 1), (u'loveliness', 2), (u'love', 101), (u'loves', 3), (u'lovely', 3), (u'lovers', 5), (u'lover', 5), (u'lovemaking', 1), (u'loved', 11)]

In [29]:
frequent_words = word_counts.filter(lambda x: x[1] > 100)

print(frequent_words.take(20))

[(u'every', 185), (u'ever', 136), (u'would', 483), (u'some', 211), (u'them', 443), (u'they', 609), (u'bennet', 307), (u'little', 192), (u'friend', 106), (u'letter', 117), (u'away', 121), (u'and', 3713), (u'had', 1185), (u'been', 536), (u'make', 171), (u'the', 4655), (u'family', 153), (u'dear', 163), (u'for', 1086), (u'illustration', 163)]

In [30]:
sorted_words = word_counts.sortByKey()

print(sorted_words.take(20))

[(u'abatement', 1), (u'abbey', 1), (u'abhorrence', 6), (u'abhorrent', 1), (u'abide', 1), (u'abiding', 1), (u'abilities', 6), (u'ability', 1), (u'able', 54), (u'ablution', 1), (u'abode', 8), (u'abominable', 6), (u'abominably', 4), (u'abominate', 2), (u'abound', 1), (u'about', 126), (u'above', 22), (u'abroad', 4), (u'abrupt', 1), (u'abruptly', 2)]

# Section C – Log File Analysis

In [32]:
log_rdd = sc.textFile("/user/hadoop/Nasa.txt")

print(log_rdd.take(5))
print("Number of partitions:", log_rdd.getNumPartitions())

[u'NASA-HTTP', u'Description', u"These two traces contain two month's worth of all HTTP requests to the NASA Kennedy Space Center WWW server in Florida.", u'Format', u'The logs are an ASCII file with one line per request, with the following columns:']
('Number of partitions:', 2)

In [34]:
log_rdd.glom().collect()

[[u'NASA-HTTP', u'Description', u"These two traces contain two month's worth of all HTTP requests to the NASA Kennedy Space Center WWW server in Florida.", u'Format', u'The logs are an ASCII file with one line per request, with the following columns:', u'host making the request. A hostname when possible, otherwise the Internet address if the name could not be looked up.', u'timestamp in the format "DAY MON DD HH:MM:SS YYYY", where DAY is the day of the week, MON is the name of the month, DD is the day of the month, HH:MM:SS is the time of day using a 24-hour clock, and YYYY is the year. The timezone is -0400.', u'request given in quotes.', u'HTTP reply code.', u'bytes in the reply.', u'Measurement', u'The first log was collected from 00:00:00 July 1, 1995 through 23:59:59 July 31, 1995, a total of 31 days. The second log was collected from 00:00:00 August 1, 1995 through 23:59:59 Agust 31, 1995, a total of 7 days. In this two week period there were 3,461,612 requests. Timestamps have 1

In [37]:
clients = log_rdd.map(lambda line: line.split()[0])

print(clients.take(10))

[u'NASA-HTTP', u'Description', u'These', u'Format', u'The', u'host', u'timestamp', u'request', u'HTTP', u'bytes']

In [41]:
log_rdd = sc.textFile("/user/hadoop/Nasa.txt")

clients = log_rdd \
    .filter(lambda line: len(line.strip()) > 0) \
    .filter(lambda line: len(line.split()) >= 1) \
    .map(lambda line: line.split()[0])

client_pairs = clients.map(lambda client: (client, 1))

client_counts = client_pairs.reduceByKey(lambda a, b: a + b)

print(client_counts.take(10))

[(u'Available', 1), (u'HTTP', 1), (u'These', 1), (u'bytes', 1), (u'request', 1), (u'Related', 1), (u'This', 1), (u'host', 1), (u'The', 5), (u'Distribution', 1)]

In [42]:
client_pairs = clients.map(lambda client: (client, 1))

client_counts = client_pairs.reduceByKey(lambda a, b: a + b)

print(client_counts.take(10))

[(u'Available', 1), (u'HTTP', 1), (u'These', 1), (u'bytes', 1), (u'request', 1), (u'Related', 1), (u'This', 1), (u'host', 1), (u'The', 5), (u'Distribution', 1)]

In [43]:
top_clients = client_counts.takeOrdered(10, key=lambda x: -x[1])

for client, count in top_clients:
    print(client, count)

(u'The', 5)
(u'Available', 1)
(u'HTTP', 1)
(u'These', 1)
(u'bytes', 1)
(u'request', 1)
(u'Related', 1)
(u'This', 1)
(u'host', 1)
(u'Distribution', 1)

In [44]:
clients = log_rdd.map(lambda line: line.split()[0])

print(clients.take(10))

[u'NASA-HTTP', u'Description', u'These', u'Format', u'The', u'host', u'timestamp', u'request', u'HTTP', u'bytes']

# Section D – Set Operations

In [46]:
rdd1 = sc.parallelize([1, 2, 3, 4, 5])
rdd2 = sc.parallelize([4, 5, 6, 7, 8])

In [47]:
union_rdd = rdd1.union(rdd2)

print("Union result:", union_rdd.collect())

('Union result:', [1, 2, 3, 4, 5, 4, 5, 6, 7, 8])

In [48]:
intersection_rdd = rdd1.intersection(rdd2)

print("Intersection result:", intersection_rdd.collect())

('Intersection result:', [4, 5])

In [49]:
unique_union = rdd1.union(rdd2).distinct()

print("Unique union:", unique_union.collect())

('Unique union:', [8, 4, 1, 5, 2, 6, 3, 7])

# Section E – Data Storage and Retrieval

In [52]:
word_counts.saveAsTextFile("/user/hadoop/output_word_counts.txt")

In [53]:
client_counts.saveAsTextFile("/user/hadoop/output_client_counts.txt")

In [55]:
saved_word_counts = sc.textFile("/user/hadoop/output_word_counts.txt")

print(saved_word_counts.take(10))

[u"(u'cruscans', 2)", u"(u'expostulation', 1)", u"(u'desirable', 11)", u"(u'foul', 1)", u"(u'four', 36)", u"(u'brightening', 1)", u"(u'protest', 2)", u"(u'sleep', 3)", u"(u'greatgrandmother', 1)", u"(u'consists', 2)"]

In [56]:
saved_client_counts = sc.textFile("/user/hadoop/output_client_counts.txt")

print(saved_client_counts.take(10))

[u"(u'Available', 1)", u"(u'HTTP', 1)", u"(u'These', 1)", u"(u'bytes', 1)", u"(u'request', 1)", u"(u'Related', 1)", u"(u'This', 1)", u"(u'host', 1)", u"(u'The', 5)", u"(u'Distribution', 1)"]